In [1]:
from torchvision.datasets import MNIST
import torchvision.transforms as transforms

trans = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_data = MNIST(root='./data',train=True,download=True,transform=trans)
test_data = MNIST(root='./data',train=False,download=True,transform=trans)

In [2]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_data,batch_size=64,shuffle=True)
test_loader = DataLoader(test_data,batch_size=64,shuffle=False)

In [3]:
import torch.nn as nn

class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.convo_layers = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), 

            nn.Conv2d(16, 32, kernel_size=3), 
            nn.ReLU(),
            nn.MaxPool2d(2, 2)              
        )

        self.fc_layer = nn.Sequential(
            nn.Linear(32 * 5 * 5, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.convo_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layer(x)
        return x

In [4]:
model = CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

NameError: name 'device' is not defined

In [ ]:
epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0
    correct = 0
    total = 0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = torch.max(outputs,1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    accuracy = 100 * correct / total
    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss: {running_loss/len(train_loader):.4f} "
        f"Accuracy: {accuracy:.2f}%"
    )

In [ ]:
model.eval()

correct = 0

total = 0

all_labels = []

all_predictions = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs,1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predicted.cpu().numpy())

print(f"Test Accuracy : {100*correct/total:.2f}%")

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(all_labels, all_predictions)

plt.figure(figsize=(8,6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("CNN Confusion Matrix")

plt.show()

In [ ]:
class RNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.rnn = nn.RNN(
            input_size=28,
            hidden_size=128,
            num_layers=1,
            batch_first=True
        )

        self.fc = nn.Linear(128,10)

    def forward(self,x):

        x = x.squeeze(1)

        output, hidden = self.rnn(x)

        output = output[:, -1, :]

        output = self.fc(output)

        return output
        
rnn_model = RNN().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    rnn_model.parameters(),
    lr=0.001
)


outputs = model(images)
outputs = rnn_model(images)

In [5]:
import os 
print(os.getcwd())

C:\Users\manan\Handwritten-Digit-Classification
